# Prometheus Star: ARC-AGI-3 Solver (Production Edition)
**Architecture:** Bridge v15 (Neural Latent Reasoning)
**Last updated:** 2026-04-08

This notebook is streamlined for solving the official ARC-AGI-3 benchmark using the latest repository updates.

In [ ]:
# ── Colab / local setup ──────────────────────────────────────────────────────
import sys, os

if 'google.colab' in sys.modules:
    if not os.path.exists('Prometheus_v0_PoC'):
        print('Cloning Prometheus repository...')
        os.system('git clone -b wp16-notebook-only https://github.com/pmcray/Prometheus_v0_PoC.git')
    else:
        os.system('git -C Prometheus_v0_PoC pull origin wp16-notebook-only')
    os.system('pip install -q -e Prometheus_v0_PoC/')
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
else:
    sys.path.insert(0, '..')

import json
import math
import random
import time
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (16, 8)

# ── WP71 imports ─────────────────────────────────────────────────────────────
from prometheus.wp71_arc_agi3 import (
    ARC3Action, ARC3Observation, ARC3Episode,
    ARC3WorldModel, ARC3GoalInferrer, ARC3ExplorationPolicy,
    ARC3StrangeLoopAgent, ARC3Benchmark, ARC3BenchmarkReport,
    verify_wp71_exit_criteria, _SyntheticARCGame, _ACTION_TYPES,
)

print('WP71 ARC-AGI-3 module loaded.')
print(f'Canonical action types ({len(_ACTION_TYPES)}): {_ACTION_TYPES}')

In [ ]:
# ── Install arc-agi toolkit ──────────────────────────────────────────────
import subprocess, sys, os

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'arc-agi'],
    capture_output=True, text=True
)
print('arc-agi install:', 'OK' if result.returncode == 0 else result.stderr[:200])

# ── API key — Colab Secrets → env var → anonymous fallback ────────
#
# HOW TO SET YOUR API KEY IN COLAB:
#   1. Click the key icon (🔑) in the left sidebar
#   2. Click "Add new secret"
#   3. Name:  ARC_API_KEY
#   4. Value: your key from arcprize.org/platform
#   5. Toggle "Notebook access" ON
#
# No key?  Three public games work anonymously: ls20, ft09, vc33

ARC_API_KEY = ''

try:
    from google.colab import userdata
    _secret = userdata.get('ARC_API_KEY')
    if _secret:
        ARC_API_KEY = _secret
        print('API key loaded from Colab Secrets.')
except Exception:
    pass   # not in Colab, or secret not set

if not ARC_API_KEY:
    ARC_API_KEY = os.environ.get('ARC_API_KEY', '')
    if ARC_API_KEY:
        print('API key loaded from environment variable.')

if ARC_API_KEY:
    os.environ['ARC_API_KEY'] = ARC_API_KEY
    print(f'API key active ({ARC_API_KEY[:6]}...)')
else:
    print('No API key found — anonymous access (3 public games: ls20, ft09, vc33)')
    print('Full access: add ARC_API_KEY to Colab Secrets (key icon in left sidebar)')
    print('             or set os.environ["ARC_API_KEY"] before running.')


In [ ]:
# ── Run Prometheus on ARC-AGI-3 ────────────
#
# Bridge v14: Dynamic game loading + CNN pattern recognition.

SELECTED_GAMES = ["ls20", "ft09", "vc33"]
if ARC_API_KEY and TOOLKIT_AVAILABLE:
    try:
        arc = arc_agi.Arcade()
        all_envs = arc.get_environments()
        SELECTED_GAMES = [e.game_id for e in all_envs]
        print(f"API Key detected: Loading all {len(SELECTED_GAMES)} games.")
    except:
        print("API Key failed: Falling back to public games.")

N_WINDOWS     = 60
WINDOW_STEPS  = 200

live_results = {}
t_start = time.time()

# Run on first 5 games to keep demo snappy if all loaded
GAMES_TO_RUN = SELECTED_GAMES[:5] if len(SELECTED_GAMES) > 3 else SELECTED_GAMES

for game_id in GAMES_TO_RUN:
    print() # safe newline
    print('=' * 55)
    print(f"  Game: {game_id}")
    print('=' * 55)
    result = run_live_game(game_id=game_id, n_windows=N_WINDOWS, window_steps=WINDOW_STEPS, mutation_rate=0.10, fitness_threshold=0.5, verbose=True)
    if result:
        live_results[game_id] = result
        sr_val = result.get('solve_rate', 0)
        ms_val = result.get('mean_score', 0)
        print(f"  --> solve rate: {sr_val:.0%}  mean score: {ms_val:.3f}")

print()
print(f"Total time: {time.time()-t_start:.1f}s")
